In [ ]:
import sys
import subprocess

required = ["transformers", "datasets", "scipy", "pandas", "torch"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "cross-encoder/stsb-distilroberta-base"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
hard_min = 2.0
hard_max = 3.0
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "hard_subset_label_range": [hard_min, hard_max],
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)
hard_df = df[(df["label"] >= hard_min) & (df["label"] <= hard_max)].reset_index(drop=True)

print({
    "num_validation_examples": int(len(df)),
    "num_hard_subset_examples": int(len(hard_df)),
    "columns": hard_df.columns.tolist(),
})
print(hard_df.head())


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print({
    "loaded_model": model_name,
    "num_labels": int(model.config.num_labels),
    "problem_type": getattr(model.config, "problem_type", None),
    "device": device,
})


In [ ]:
sent1 = hard_df["sentence1"].tolist()
sent2 = hard_df["sentence2"].tolist()

all_scores = []

with torch.no_grad():
    for start in range(0, len(hard_df), batch_size):
        end = min(start + batch_size, len(hard_df))
        batch_s1 = sent1[start:end]
        batch_s2 = sent2[start:end]
        enc = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        outputs = model(**enc)
        logits = outputs.logits
        if logits.ndim == 2 and logits.shape[-1] == 1:
            batch_scores = logits.squeeze(-1)
        else:
            batch_scores = logits.squeeze()
        all_scores.append(batch_scores.detach().float().cpu())

pred_scores = torch.cat(all_scores).numpy().astype(np.float32)
pred_scores = np.clip(pred_scores, 0.0, 5.0)

print(pd.DataFrame({
    "label": hard_df["label"].head(10).to_numpy(),
    "pred_score": pred_scores[:10],
} ))


In [ ]:
labels = hard_df["label"].to_numpy(dtype=np.float32)
signed_error = pred_scores - labels
abs_error = np.abs(signed_error)

pearson_value = pearsonr(pred_scores, labels).statistic
spearman_value = spearmanr(pred_scores, labels).statistic
mae_value = abs_error.mean()
bias_value = signed_error.mean()

results_df = hard_df.copy()
results_df["pred_score"] = pred_scores
results_df["signed_error"] = signed_error.astype(np.float32)
results_df["abs_error"] = abs_error.astype(np.float32)
results_df["error_direction"] = np.where(results_df["signed_error"] > 0, "overestimate", np.where(results_df["signed_error"] < 0, "underestimate", "exact"))

print(results_df[["sentence1", "sentence2", "label", "pred_score", "signed_error", "abs_error", "error_direction"]].head(10))


In [ ]:
signed_error_table = results_df[[
    "sentence1", "sentence2", "label", "pred_score", "signed_error", "abs_error", "error_direction"
]].copy()

print("most_overestimated_examples")
print(signed_error_table.sort_values("signed_error", ascending=False).head(15).to_string(index=False))

print("most_underestimated_examples")
print(signed_error_table.sort_values("signed_error", ascending=True).head(15).to_string(index=False))

print("largest_absolute_errors")
print(signed_error_table.sort_values("abs_error", ascending=False).head(15).to_string(index=False))


In [ ]:
direction_summary = results_df.groupby("error_direction").agg(
    count=("signed_error", "size"),
    mean_signed_error=("signed_error", "mean"),
    mean_abs_error=("abs_error", "mean"),
    mean_gold_label=("label", "mean"),
    mean_pred_score=("pred_score", "mean"),
).reset_index()

print(direction_summary.to_string(index=False))

runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"hard_subset_gold_label_range: [{hard_min}, {hard_max}]")
print(f"num_examples_full_validation: {len(df)}")
print(f"num_examples_hard_subset: {len(hard_df)}")
print(f"pearson_pred_score: {pearson_value:.6f}")
print(f"spearman_pred_score: {spearman_value:.6f}")
print(f"mae_pred_score: {mae_value:.6f}")
print(f"mean_signed_error_bias: {bias_value:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
